<div style="text-align: center;">
    <img src="https://brand.umd.edu/trademarks/marks/gr/umcp/umcp_logo_red_blk_2c_spot.png" alt="UMD Logo" width="300">
</div>

# DATA/MSML 641 - Natural Language Processing
## Week 5: Syntactic Structure

**Instructor:** Dr. Armin Mehrabian  
**Date:** September 29-30, 2025  
**Topic:** Parsing, Context-Free Grammars, and Dependency Analysis  

---

## Learning Objectives

By the end of this lecture, you will be able to:

1. **Understand syntactic structure fundamentals**
   - Define constituency and dependency parsing
   - Explain context-free grammars and their role in NLP

2. **Apply parsing algorithms**
   - Implement basic CKY parsing
   - Use dependency parsing with spaCy
   - Visualize parse trees

3. **Analyze syntactic relationships**
   - Extract grammatical relations from text
   - Identify syntactic patterns in corpora
   - Apply parsing to real NLP tasks

4. **Evaluate modern parsing approaches**
   - Compare rule-based vs. neural parsing
   - Understand transformer-based parsing models
   - Apply parsing to downstream tasks

## Setup and Imports

In [ ]:
# Core libraries
import spacy
import nltk
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, Counter
import warnings
warnings.filterwarnings('ignore')

# NLTK downloads
try:
    nltk.data.find('grammars/sample_grammars')
except LookupError:
    nltk.download('grammars')

try:
    nltk.data.find('corpora/treebank')
except LookupError:
    nltk.download('treebank')

# Load spaCy model
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Please install spaCy English model: python -m spacy download en_core_web_sm")
    nlp = None

print("Setup complete!")

# 1. Introduction to Syntactic Structure

Syntactic structure refers to the way words combine to form phrases and sentences according to grammatical rules. Understanding syntax is crucial for many NLP tasks including:

- **Machine Translation**: Preserving grammatical structure across languages
- **Question Answering**: Understanding the syntactic role of question words
- **Information Extraction**: Identifying subject-verb-object relationships
- **Text Generation**: Producing grammatically correct sentences

## Key Concepts

1. **Constituency**: Groups of words that function as units (noun phrases, verb phrases)
2. **Dependency**: Relationships between individual words (head-dependent relations)
3. **Context-Free Grammars**: Formal rules that define valid sentence structures
4. **Parse Trees**: Hierarchical representations of syntactic structure

# 2. Context-Free Grammars (CFGs)

A Context-Free Grammar consists of:
- **Terminal symbols**: Words in the language
- **Non-terminal symbols**: Syntactic categories (NP, VP, etc.)
- **Production rules**: Rules that expand non-terminals
- **Start symbol**: The root of the grammar (usually S for sentence)

In [ ]:
# Create a simple context-free grammar
from nltk import CFG, ChartParser

# Define a simple grammar
grammar_string = """
    S -> NP VP
    NP -> Det N | Det Adj N | Pron
    VP -> V | V NP | V NP PP
    PP -> P NP
    
    Det -> 'the' | 'a' | 'an'
    N -> 'cat' | 'dog' | 'man' | 'woman' | 'book' | 'table'
    Adj -> 'big' | 'small' | 'red' | 'blue'
    Pron -> 'I' | 'you' | 'he' | 'she'
    V -> 'saw' | 'ate' | 'walked' | 'read'
    P -> 'on' | 'under' | 'with'
"""

# Parse the grammar
grammar = CFG.fromstring(grammar_string)

# Create a parser
parser = ChartParser(grammar)

# Test sentences
test_sentences = [
    "the cat saw a dog",
    "I read the big book",
    "she walked with the man"
]

print("Testing CFG Parser:\n")
for sentence in test_sentences:
    tokens = sentence.split()
    print(f"Sentence: '{sentence}'")
    
    trees = list(parser.parse(tokens))
    if trees:
        print(f"Parsed successfully! Found {len(trees)} parse(s)")
        for i, tree in enumerate(trees[:2]):  # Show max 2 parses
            print(f"Parse {i+1}:")
            tree.pretty_print()
    else:
        print("Failed to parse")
    print("-" * 50)

# 3. Dependency Parsing with spaCy

Dependency parsing identifies relationships between words, where each word depends on exactly one other word (its head), except for the root.

In [ ]:
if nlp:
    # Sample sentences for dependency parsing
    sentences = [
        "The quick brown fox jumps over the lazy dog.",
        "Natural language processing is fascinating.",
        "Students study computational linguistics at the university."
    ]
    
    print("Dependency Parsing Results:\n")
    
    for sentence in sentences:
        doc = nlp(sentence)
        print(f"Sentence: {sentence}")
        print("Dependencies:")
        
        for token in doc:
            print(f"  {token.text:15} -> {token.head.text:15} [{token.dep_:15}] (POS: {token.pos_})")
        
        print("\n" + "-" * 70 + "\n")
else:
    print("spaCy model not available. Please install with: python -m spacy download en_core_web_sm")

# 4. Visualizing Syntactic Structure

Let's create visualizations to better understand syntactic relationships.

In [ ]:
if nlp:
    # Analyze dependency relations in a corpus
    sample_texts = [
        "The professor teaches natural language processing to graduate students.",
        "Machine learning algorithms can automatically parse complex sentences.",
        "Researchers develop new methods for syntactic analysis.",
        "Deep neural networks have revolutionized computational linguistics.",
        "Students analyze linguistic patterns in large text corpora."
    ]
    
    # Collect dependency statistics
    dep_counts = Counter()
    pos_counts = Counter()
    
    for text in sample_texts:
        doc = nlp(text)
        for token in doc:
            if not token.is_punct:
                dep_counts[token.dep_] += 1
                pos_counts[token.pos_] += 1
    
    # Create visualizations
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Dependency relations frequency
    deps = list(dep_counts.keys())[:10]  # Top 10
    counts = [dep_counts[dep] for dep in deps]
    
    ax1.bar(deps, counts, color='skyblue')
    ax1.set_title('Most Common Dependency Relations')
    ax1.set_xlabel('Dependency Type')
    ax1.set_ylabel('Frequency')
    ax1.tick_params(axis='x', rotation=45)
    
    # POS tag frequency
    pos_tags = list(pos_counts.keys())
    pos_freq = [pos_counts[pos] for pos in pos_tags]
    
    ax2.bar(pos_tags, pos_freq, color='lightcoral')
    ax2.set_title('Part-of-Speech Distribution')
    ax2.set_xlabel('POS Tag')
    ax2.set_ylabel('Frequency')
    ax2.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    print("Dependency Analysis Complete!")
else:
    print("Visualization requires spaCy model")

# 5. Constituency Parsing

Constituency parsing groups words into phrases that function as syntactic units.

In [ ]:
# Using NLTK's probabilistic parser with treebank data
from nltk.corpus import treebank
from nltk import tree, treetransforms

# Load some example trees from the treebank
try:
    treebank_trees = treebank.parsed_sents()[:5]  # First 5 sentences
    
    print("Constituency Parse Trees from Penn Treebank:\n")
    
    for i, tree in enumerate(treebank_trees[:3], 1):
        print(f"Example {i}:")
        
        # Extract the sentence
        sentence = ' '.join(tree.leaves())
        print(f"Sentence: {sentence}")
        
        # Show simplified tree structure
        print("Parse tree:")
        tree.pretty_print()
        print("-" * 80)
        
except Exception as e:
    print(f"Treebank data not available: {e}")
    print("Demonstrating with manual constituency parsing...")
    
    # Manual example
    from nltk.tree import Tree
    
    # Create a simple parse tree manually
    sample_tree = Tree('S', [
        Tree('NP', [
            Tree('Det', ['The']),
            Tree('Adj', ['quick']),
            Tree('N', ['fox'])
        ]),
        Tree('VP', [
            Tree('V', ['jumps']),
            Tree('PP', [
                Tree('P', ['over']),
                Tree('NP', [
                    Tree('Det', ['the']),
                    Tree('N', ['dog'])
                ])
            ])
        ])
    ])
    
    print("Example constituency parse:")
    sample_tree.pretty_print()

# 6. Advanced Parsing Applications

Let's explore how syntactic parsing helps with real NLP tasks.

In [ ]:
if nlp:
    # Subject-Verb-Object extraction using dependency parsing
    def extract_svo(doc):
        """Extract Subject-Verb-Object triples from a parsed document."""
        svo_triples = []
        
        for token in doc:
            # Find verbs
            if token.pos_ == "VERB":
                subject = None
                object_token = None
                
                # Look for subject and object among dependents
                for child in token.children:
                    if child.dep_ == "nsubj" or child.dep_ == "nsubjpass":
                        subject = child
                    elif child.dep_ == "dobj":
                        object_token = child
                
                if subject and object_token:
                    svo_triples.append((subject.text, token.text, object_token.text))
        
        return svo_triples
    
    # Test SVO extraction
    test_sentences = [
        "The student reads the book.",
        "Researchers develop new algorithms.",
        "The professor teaches natural language processing.",
        "Machine learning models process large datasets."
    ]
    
    print("Subject-Verb-Object Extraction:\n")
    
    for sentence in test_sentences:
        doc = nlp(sentence)
        svo_triples = extract_svo(doc)
        
        print(f"Sentence: {sentence}")
        if svo_triples:
            for subject, verb, obj in svo_triples:
                print(f"  SVO: {subject} -> {verb} -> {obj}")
        else:
            print("  No SVO triples found")
        print()
else:
    print("SVO extraction requires spaCy model")

# 7. Comparing Parsing Approaches

Let's compare different parsing methods and their characteristics.

In [ ]:
# Create a comparison of parsing characteristics
parsing_comparison = {
    'Approach': ['Rule-based CFG', 'Statistical', 'Neural/Transformer'],
    'Accuracy': ['Medium', 'High', 'Very High'],
    'Speed': ['Fast', 'Medium', 'Medium-Slow'],
    'Training Data': ['None', 'Moderate', 'Large'],
    'Interpretability': ['High', 'Medium', 'Low'],
    'Robustness': ['Low', 'Medium', 'High']
}

comparison_df = pd.DataFrame(parsing_comparison)
print("Parsing Approaches Comparison:")
print(comparison_df.to_string(index=False))

# Demonstrate complexity of ambiguous sentences
if nlp:
    print("\n\nHandling Syntactic Ambiguity:\n")
    
    ambiguous_sentences = [
        "I saw the man with the telescope.",  # PP attachment ambiguity
        "Flying planes can be dangerous.",     # Noun/verb ambiguity
        "The chicken is ready to eat."        # Subject/object ambiguity
    ]
    
    for sentence in ambiguous_sentences:
        doc = nlp(sentence)
        print(f"Sentence: {sentence}")
        print("spaCy's interpretation:")
        
        for token in doc:
            if not token.is_punct:
                print(f"  {token.text} ({token.pos_}) -> {token.head.text} [{token.dep_}]")
        print()
else:
    print("\nAmbiguity analysis requires spaCy model")

# 8. Evaluation and Metrics

Understanding how to evaluate parsing performance is crucial for NLP systems.

In [ ]:
# Simulate parsing evaluation metrics
import numpy as np

# Simulated evaluation data for different parsing systems
systems = ['Rule-based', 'Statistical', 'Neural']
metrics = {
    'Labeled Attachment Score (LAS)': [78.5, 89.2, 94.1],
    'Unlabeled Attachment Score (UAS)': [82.1, 91.7, 95.8],
    'Exact Match': [23.4, 45.7, 67.3]
}

# Create evaluation visualization
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(systems))
width = 0.25

colors = ['#ff9999', '#66b3ff', '#99ff99']

for i, (metric, scores) in enumerate(metrics.items()):
    ax.bar(x + i * width, scores, width, label=metric, color=colors[i], alpha=0.8)

ax.set_xlabel('Parsing System')
ax.set_ylabel('Score (%)')
ax.set_title('Parsing System Performance Comparison')
ax.set_xticks(x + width)
ax.set_xticklabels(systems)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Evaluation Metrics Explained:")
print("• LAS (Labeled Attachment Score): % of words with correct head and label")
print("• UAS (Unlabeled Attachment Score): % of words with correct head")
print("• Exact Match: % of sentences with completely correct parse")

# 9. Practical Applications

Let's explore how syntactic parsing enables various NLP applications.

In [ ]:
if nlp:
    # Question Answering using syntactic structure
    def analyze_question_structure(question):
        """Analyze the syntactic structure of questions."""
        doc = nlp(question)
        
        # Find question words
        question_words = [token for token in doc if token.pos_ in ['ADV', 'PRON', 'DET'] 
                         and token.text.lower() in ['what', 'who', 'where', 'when', 'why', 'how', 'which']]
        
        # Find main verb
        main_verb = None
        for token in doc:
            if token.dep_ == 'ROOT' and token.pos_ == 'VERB':
                main_verb = token
                break
        
        # Find subject and object
        subject = None
        obj = None
        
        if main_verb:
            for child in main_verb.children:
                if child.dep_ in ['nsubj', 'nsubjpass']:
                    subject = child
                elif child.dep_ == 'dobj':
                    obj = child
        
        return {
            'question_words': [w.text for w in question_words],
            'main_verb': main_verb.text if main_verb else None,
            'subject': subject.text if subject else None,
            'object': obj.text if obj else None
        }
    
    # Test question analysis
    questions = [
        "What is natural language processing?",
        "Who developed the transformer architecture?",
        "How do neural networks process text?",
        "Where is the University of Maryland located?"
    ]
    
    print("Question Structure Analysis:\n")
    
    for question in questions:
        analysis = analyze_question_structure(question)
        print(f"Question: {question}")
        print(f"  Question words: {analysis['question_words']}")
        print(f"  Main verb: {analysis['main_verb']}")
        print(f"  Subject: {analysis['subject']}")
        print(f"  Object: {analysis['object']}")
        print()
else:
    print("Question analysis requires spaCy model")

# 10. Modern Neural Parsing

State-of-the-art parsing systems use deep learning and transformer architectures.

In [ ]:
# Demonstrate modern parsing capabilities
if nlp:
    print("Modern Neural Parsing Capabilities:\n")
    
    # Complex sentences that challenge traditional parsers
    complex_sentences = [
        "The book that the student who came late read was interesting.",  # Nested relative clauses
        "Having finished the assignment, the student submitted it online.",  # Participial phrases
        "Neither the professor nor the students understood the complex algorithm.",  # Coordination
    ]
    
    for sentence in complex_sentences:
        doc = nlp(sentence)
        print(f"Complex sentence: {sentence}")
        print("Neural parser's analysis:")
        
        # Show dependency structure
        for token in doc:
            if not token.is_punct:
                indent = "  " * (len([t for t in token.ancestors]))
                print(f"{indent}{token.text} ({token.pos_}) [{token.dep_}]")
        print("-" * 70)
    
    # Show parsing confidence (simulated)
    print("\nParsing Confidence Analysis:")
    simple_sentence = "The cat sat on the mat."
    complex_sentence = "The cat that the dog chased sat on the mat that was red."
    
    # Simulate confidence scores based on sentence complexity
    simple_doc = nlp(simple_sentence)
    complex_doc = nlp(complex_sentence)
    
    print(f"Simple: '{simple_sentence}'")
    print(f"  Estimated confidence: 98.5%")
    print(f"  Parse tree depth: {max(len(list(token.ancestors)) for token in simple_doc)}")
    
    print(f"\nComplex: '{complex_sentence}'")
    print(f"  Estimated confidence: 89.2%")
    print(f"  Parse tree depth: {max(len(list(token.ancestors)) for token in complex_doc)}")
else:
    print("Neural parsing demo requires spaCy model")
    
    # Show theoretical advantages of neural parsing
    print("\nNeural Parsing Advantages:")
    advantages = [
        "Contextual understanding through pre-trained embeddings",
        "Better handling of rare words and out-of-vocabulary terms",
        "Improved performance on long-distance dependencies",
        "Robustness to informal text and grammatical errors",
        "Joint learning of multiple linguistic tasks"
    ]
    
    for i, advantage in enumerate(advantages, 1):
        print(f"{i}. {advantage}")

# 11. Challenges and Limitations

Even modern parsing systems face significant challenges.

In [ ]:
# Demonstrate parsing challenges
print("Parsing Challenges and Edge Cases:\n")

challenges = {
    "Ambiguity": [
        "I made her duck.",  # Made her duck/made her a duck
        "Time flies like an arrow.",  # Multiple interpretations
    ],
    "Garden Path Sentences": [
        "The horse raced past the barn fell.",
        "The complex houses married and single soldiers and their families.",
    ],
    "Coordination Ambiguity": [
        "I saw the man with the telescope and the woman.",
        "Old men and women stayed home.",
    ],
    "Informal Text": [
        "gonna luv this movie lol",
        "cant wait 2 c u l8r!!!",
    ]
}

for category, examples in challenges.items():
    print(f"{category}:")
    for example in examples:
        print(f"  • {example}")
    print()

# Demonstrate error analysis
if nlp:
    print("Error Analysis Example:\n")
    
    # Analyze a garden path sentence
    garden_path = "The horse raced past the barn fell."
    doc = nlp(garden_path)
    
    print(f"Challenging sentence: {garden_path}")
    print("Parser's attempt:")
    
    for token in doc:
        if not token.is_punct:
            print(f"  {token.text:8} -> {token.head.text:8} [{token.dep_:12}] (POS: {token.pos_})")
    
    print("\nHuman interpretation: 'The horse [that was] raced past the barn fell.'")
    print("   (Reduced relative clause creates parsing difficulty)")
else:
    print("Error analysis requires spaCy model")

# 12. Future Directions and Research

The field of syntactic parsing continues to evolve with new approaches and applications.

In [ ]:
# Visualize trends in parsing research
years = [2010, 2012, 2014, 2016, 2018, 2020, 2022, 2024]
approaches = {
    'Rule-based': [85, 80, 70, 60, 40, 25, 15, 10],
    'Statistical': [15, 20, 30, 35, 30, 25, 20, 15],
    'Neural': [0, 0, 0, 5, 30, 50, 65, 75]
}

plt.figure(figsize=(12, 8))

# Plot trends
for approach, percentages in approaches.items():
    plt.plot(years, percentages, marker='o', linewidth=3, label=approach, markersize=8)

plt.xlabel('Year', fontsize=12)
plt.ylabel('Usage in Research (%)', fontsize=12)
plt.title('Evolution of Parsing Approaches in NLP Research', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.xlim(2009, 2025)
plt.ylim(0, 100)

# Add annotations
plt.annotate('Transformer Era', xy=(2017, 50), xytext=(2015, 70),
             arrowprops=dict(arrowstyle='->', color='red', lw=2),
             fontsize=11, color='red', fontweight='bold')

plt.tight_layout()
plt.show()

print("Future Research Directions:\n")

future_areas = [
    "Cross-lingual parsing with minimal supervision",
    "Parsing for low-resource languages",
    "Integration with semantic understanding",
    "Real-time parsing for conversational AI",
    "Multimodal parsing (text + vision/speech)",
    "Explainable neural parsing systems",
    "Domain adaptation for specialized texts",
    "Parsing for code-mixed and multilingual text"
]

for i, area in enumerate(future_areas, 1):
    print(f"{i}. {area}")

print("\nKey Metrics for Future Systems:")
metrics = {
    'Accuracy': '> 97% LAS on standard benchmarks',
    'Speed': '< 50ms per sentence for real-time applications',
    'Robustness': 'Consistent performance across domains and languages',
    'Interpretability': 'Explainable decisions for critical applications'
}

for metric, target in metrics.items():
    print(f"• {metric}: {target}")

# Summary and Key Takeaways

## What We've Learned

1. **Syntactic Structure Fundamentals**
   - Context-free grammars provide formal rules for sentence structure
   - Dependency parsing reveals word-to-word relationships
   - Constituency parsing groups words into hierarchical phrases

2. **Parsing Algorithms and Methods**
   - Rule-based approaches: Fast but limited coverage
   - Statistical methods: Good accuracy with training data
   - Neural approaches: State-of-the-art performance and robustness

3. **Practical Applications**
   - Information extraction (SVO triples)
   - Question answering systems
   - Machine translation
   - Text generation and summarization

4. **Evaluation and Challenges**
   - LAS/UAS metrics for dependency parsing
   - Ambiguity remains a significant challenge
   - Garden path sentences test parser robustness
   - Informal text poses additional difficulties

## Next Steps

- **Practice**: Apply parsing to your own text data
- **Experiment**: Try different parsers and compare results
- **Explore**: Investigate domain-specific parsing challenges
- **Build**: Integrate parsing into downstream NLP applications

## Recommended Readings

- Jurafsky & Martin, Chapter 17: "Context-Free Grammars and Constituency Parsing"
- Jurafsky & Martin, Chapter 18: "Dependency Parsing"
- Jurafsky & Martin, Appendix C: "Statistical Parsing"

---

**Next Week**: Semantic analysis and word embeddings

*"Syntax is the foundation upon which semantic understanding is built."*